# CSIRO Image2Biomass — CNN & DINOv2 Training (Google Colab)

**Master's in Green Data Science — Practical ML**

Trains all image models (SimpleCNN, ResNet18 frozen/fine-tuned, DINOv2 frozen)
on GPU and compares them with the tabular baseline. Code is pulled from GitHub;
data is downloaded from a public link with `gdown` (no personal Drive needed).

**Before running:** Runtime -> Change runtime type -> GPU (T4), then Run all.

## Pipeline order
1. GPU check
2. Clone code from GitHub
3. Download data (gdown)
4. Import project
5. Smoke test
6. Train all image models (CNN + ResNet) -> `results`
7. Train DINOv2 -> add to `results`
8. Full comparison (all models) + per-target R2 + Wilcoxon
9. Save metrics + deployable checkpoint

## 1. GPU check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone code from GitHub

Pulls the latest code. Safe to re-run: clones if absent, otherwise `git pull`.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Shuhrat-1/image2biomass.git"
PROJECT = Path("/content/image2biomass")

if PROJECT.exists():
    !cd {PROJECT} && git pull
else:
    !git clone {REPO_URL} {PROJECT}

print("contents:", os.listdir(PROJECT))

## 3. Download data (public, no personal Drive needed)

Images and labels are hosted as public Google Drive files and fetched with
`gdown`. Anyone running this notebook (including the instructor) gets the same
data without mounting their own Drive.

In [ ]:
import zipfile
from pathlib import Path

!pip install -q gdown

ZIP_FILE_ID = "1x2OaQ1GUIk2fwmFfXBcMg0SsmlN7ZwxB"   # labelled.zip
CSV_FILE_ID = "1s5y4JAp9x7lDKl4gFHmuRczDIWi-5l3u"   # labelled.csv

RAW = PROJECT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
(PROJECT / "data" / "processed").mkdir(parents=True, exist_ok=True)

if not (RAW / "labelled.csv").exists():
    import gdown
    gdown.download(id=CSV_FILE_ID, output=str(RAW / "labelled.csv"), quiet=False)

if not (RAW / "labelled").exists():
    import gdown
    zip_path = PROJECT / "labelled.zip"
    gdown.download(id=ZIP_FILE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW)

n_jpg = len(list((RAW / "labelled").glob("*.jpg")))
print("labelled images:", n_jpg)
print("labelled.csv exists:", (RAW / "labelled.csv").exists())

If `labelled images: 0`, the zip contains a nested folder. Inspect with
`!ls /content/image2biomass/data/raw` and move the .jpg files so they sit
directly under `data/raw/labelled/`.

## 4. Import project

In [ ]:
import sys, importlib
sys.path.insert(0, str(PROJECT / 'src'))

import config, eda, splits, dataset, model, train, baseline
for m in (config, eda, splits, dataset, model, train, baseline):
    importlib.reload(m)

print("in_colab:", config.in_colab())
print("LABELLED_CSV exists:", config.LABELLED_CSV.exists())
print("device:", train.get_device())

## 5. Load data and smoke test

Quick 5-epoch run to confirm the GPU pipeline works before the full sweep.

In [ ]:
wide = eda.group_rare_species(eda.to_wide(eda.load_long()))
print("wide:", wide.shape)

smoke = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    n_splits=5, batch_size=32, num_workers=2,
    train_kwargs={"max_epochs": 5, "patience": 3}, verbose=True,
)
print("smoke weighted RMSE:", round(smoke["weighted_rmse_mean"], 3))

## 6. Train all image models (CNN + ResNet)

- **simple_cnn**: from scratch (session 10)
- **resnet18 frozen**: feature extraction (session 12)
- **resnet18 fine-tune**: full fine-tuning (session 12), smaller LR

All results go into the `results` dict.

In [ ]:
common = dict(n_splits=5, img_size=224, batch_size=32, num_workers=2, verbose=True)
results = {}

results["simple_cnn"] = train.cross_validate(
    wide, kind="simple_cnn", model_kwargs={},
    train_kwargs={"max_epochs": 60, "patience": 10, "lr": 1e-3}, **common)

results["resnet18_frozen"] = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    train_kwargs={"max_epochs": 40, "patience": 8, "lr": 1e-3}, **common)

results["resnet18_finetune"] = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": False},
    train_kwargs={"max_epochs": 30, "patience": 8, "lr": 1e-4}, **common)

for name, r in results.items():
    print(f"{name:22} weighted RMSE = {r['weighted_rmse_mean']:.3f} "
          f"+/- {r['weighted_rmse_std']:.3f}")

## 7. Train DINOv2 (frozen, self-supervised)

DINOv2 frozen vs ResNet frozen isolates the effect of pretraining
(self-supervised vs ImageNet) at the same regime. Embeddings are precomputed
once and cached; only a small head is trained per fold. Added to `results` so the
comparison below includes it.

In [ ]:
config.RUN_DINOV2_REGRESSION = True

if config.RUN_DINOV2_REGRESSION:
    results["dinov2_frozen"] = train.cross_validate_dinov2(wide, n_splits=5, verbose=True)
    r = results["dinov2_frozen"]
    print(f"dinov2_frozen          weighted RMSE = {r['weighted_rmse_mean']:.3f} "
          f"+/- {r['weighted_rmse_std']:.3f}")

## 8. Full comparison — all models

Now that every model (including DINOv2) is in `results`, we build:
(a) the comparison table with mean +/- std,
(b) per-target R2 for **every** model,
(c) a paired Wilcoxon test (RF vs the best image model).

In [ ]:
# (a) Comparison table: tabular (computed) + all image models, mean +/- std
import pandas as pd, numpy as np
from scipy.stats import wilcoxon

folds = splits.get_cv_folds(wide, n_splits=5)
tab_all = pd.concat(
    [baseline.evaluate_naive(wide, folds),
     baseline.evaluate_model(wide, folds, "rf")], ignore_index=True)
tab_wpf = baseline.weighted_per_fold(tab_all)

all_models = {
    "RandomForest (tabular)": tab_wpf["rf"],
    "naive_median (tabular)": tab_wpf["naive_median"],
}
for name, r in results.items():
    all_models[f"{name} (image)"] = {
        "weighted_rmse_mean": r["weighted_rmse_mean"],
        "weighted_rmse_std": r["weighted_rmse_std"],
        "fold_weighted": r["fold_weighted"],
    }

rows = [{"model": m,
         "weighted_rmse": f"{d['weighted_rmse_mean']:.2f} ± {d['weighted_rmse_std']:.2f}",
         "_sort": d["weighted_rmse_mean"]} for m, d in all_models.items()]
comparison = pd.DataFrame(rows).sort_values("_sort").drop(columns="_sort").reset_index(drop=True)
print(comparison.to_string(index=False))

In [ ]:
# (b) Per-target R2(Total) for every image model (robust: read by position)
print("Per-target R2 on Dry_Total_g:")
for name, r in results.items():
    summ = r["summary"]
    # summary is indexed by (model, target); find the Dry_Total_g row regardless of model label
    total_rows = summ.xs("Dry_Total_g", level="target")
    r2_total = float(total_rows[("r2", "mean")].iloc[0])
    print(f"  {name:20} R2(Total) = {r2_total:.3f}")

In [ ]:
# (c) Paired Wilcoxon: RandomForest vs best image model
rf_folds = all_models["RandomForest (tabular)"]["fold_weighted"]
best_image = min((m for m in all_models if "(image)" in m),
                 key=lambda m: all_models[m]["weighted_rmse_mean"])
img_folds = all_models[best_image]["fold_weighted"]
print(f"Best image model: {best_image}")
print(f"RF per-fold:  {[round(x,2) for x in rf_folds]}")
print(f"{best_image}:  {[round(x,2) for x in img_folds]}")
stat, p = wilcoxon(rf_folds, img_folds)
print(f"\nWilcoxon RF vs {best_image}: statistic={stat}, p={p:.4f}")
print("Caveat: 5 folds, grouped by image (not fully independent) — p is indicative.")

In [ ]:
# (d) Full per-target summary for the best image model (detailed table)
print("Detailed per-target metrics —", best_image)
results[best_image.replace(" (image)", "")]["summary"]

## 9. Save metrics and a deployable checkpoint

Saves the comparison + trains a final image-only ResNet on (almost) all data for
the Gradio app (lightweight backbone chosen for deployment).

In [ ]:
import json
OUTPUTS = PROJECT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

out = {name: {"weighted_rmse_mean": r["weighted_rmse_mean"],
              "weighted_rmse_std": r["weighted_rmse_std"]}
       for name, r in results.items()}
with open(OUTPUTS / "cnn_results.json", "w") as f:
    json.dump(out, f, indent=2)
comparison.to_csv(OUTPUTS / "model_comparison.csv", index=False)
print("metrics saved to", OUTPUTS)

In [ ]:
# Final deployable model (image-only fine-tuned ResNet) for the Gradio app
final_model, info = train.train_final_model(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": False},
    train_kwargs={"max_epochs": 30, "patience": 8, "lr": 1e-4},
)
ckpt_path = OUTPUTS / "best_model.pt"
train.save_checkpoint(final_model, info, ckpt_path)
print("checkpoint saved to", ckpt_path)
print("val weighted RMSE:", round(info["val_weighted_rmse"], 3))
print("Download best_model.pt from the Colab file browser for the Gradio app.")

## Notes / conclusions

- **RandomForest on metadata (11.83) is the best model overall**, beating the
  best image model, DINOv2 frozen (16.50), by ~28% — cheap ground measurements
  (NDVI + height) outperform imagery on 357 samples.
- **Pretraining quality matters**: frozen DINOv2 (16.50, self-supervised) far
  exceeds frozen ResNet (23.05, ImageNet) and matches fine-tuned ResNet (16.83).
- Numbers vary slightly between runs (head init, batch order); use one run for
  the report. Download `best_model.pt` for the Gradio app (see DEPLOY.md).